# 👑 QUILLAN-RONIN v5.4-ONI — INTERACTIVE MASTER TRAINING WORKBENCH
=============================================================================
**Unified Sovereign Architecture | BitNet 1.58b STE | MoBA Long-Context | MuonK2 Optimizer**

This notebook provides an interactive training and evaluation workbench directly inside the IDE.
- Live in-cell loss tracking and progress monitoring
- High-Density Anchor Curriculum to drive loss down to **sub-3.0**
- Real-time generation probes and interactive prompt testing


In [ ]:
# [Cell 1] Environment, Hardware Governor & Auto-Detect (Colab / Local)
import os
import sys
import time
import math
import json
import logging
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# Configure UTF-8 for notebook stdout across Windows & Linux
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

# Auto-detect Local Windows IDE vs Google Colab Cloud
if Path("C:/02_QUILLAN").exists():
    IN_COLAB = False
    REPO_ROOT = Path("C:/02_QUILLAN")
    print("[OK] Running locally in IDE on Windows (Repository: C:/02_QUILLAN)!")
elif Path("/content").exists():
    IN_COLAB = True
    print("[OK] Running in Google Colab Cloud GPU environment!")
    REPO_ROOT = Path("/content/Quillan-Ronin")
    if not REPO_ROOT.exists():
        print("Cloning Quillan-Ronin repository into Colab...")
        import subprocess
        subprocess.run(["git", "clone", "https://github.com/leeex1/Quillan-Ronin.git", str(REPO_ROOT)], check=True)
        print("[OK] Repository cloned successfully!")
else:
    IN_COLAB = False
    REPO_ROOT = Path.cwd()
    print(f"[OK] Running locally from working directory: {REPO_ROOT}")

for p in [
    REPO_ROOT / "09 - Projects" / "projects" / "oni",
    REPO_ROOT / "03 - Training & Model" / "scripts",
    REPO_ROOT / "scripts",
    REPO_ROOT,
]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Hardware Governor & Device Selection
# On Windows IDE with local GTX 1050 (sm_61), force CPU to prevent CUDA kernel mismatch.
# On Linux / Google Colab cloud, use CUDA GPU acceleration.
if sys.platform == "win32" and Path("C:/02_QUILLAN").exists():
    device = torch.device("cpu")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cpu":
    try:
        torch.set_num_threads(min(2, os.cpu_count() or 2))
        try:
            torch.set_num_interop_threads(min(2, os.cpu_count() or 2))
        except Exception:
            pass
    except Exception:
        pass

try:
    import psutil
    p = psutil.Process()
    if hasattr(psutil, "BELOW_NORMAL_PRIORITY_CLASS"):
        p.nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass

print(f"[OK] PyTorch version: {torch.__version__}")
dev_desc = torch.cuda.get_device_name(0) if device.type == "cuda" else f"CPU ({torch.get_num_threads()} threads, Zero-Lag Mode)"
print(f"[OK] Active Device: {device} [{dev_desc}]")
print(f"[OK] Repository Root: {REPO_ROOT} (Found: {REPO_ROOT.exists()})")


In [ ]:
# [Cell 2] Load Canonical Architecture & Checkpoint
import sys
for p in [
    REPO_ROOT / "09 - Projects" / "projects" / "oni",
    REPO_ROOT / "03 - Training & Model" / "scripts",
    REPO_ROOT / "scripts",
    REPO_ROOT,
]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from quillan_v5_4_oni import QuillanRoninOni, QuillanOniConfig
from sovereign_inference_engine import SovereignTokenizer
from quillan_muonk2_optimizer import create_quillan_muonk2_optimizer

tokenizer = SovereignTokenizer("gpt2")

# Dynamically construct config kwargs to maintain compatibility across all environments
cfg_kwargs = {
    "n_layer": 6,
    "hidden_dim": 1024,
    "max_seq_len": 256,
    "num_experts": 34,
    "router_mode": "dense_pull",
}

# Dynamically check if MoBA is supported in the loaded QuillanOniConfig definition
if hasattr(QuillanOniConfig, "use_moba") or "use_moba" in getattr(QuillanOniConfig, "__dataclass_fields__", {}):
    cfg_kwargs["use_moba"] = True
    cfg_kwargs["moba_block_size"] = 64
    cfg_kwargs["moba_top_k"] = 2
    print("[OK] Moonshot AI MoBA (Mixture of Block Attention) active!")
else:
    print("[INFO] Remote config uses canonical attention (MoBA optional flags omitted).")

cfg = QuillanOniConfig(**cfg_kwargs)

print("[1/2] Instantiating Quillan-Ronin v5.4-ONI architecture...")
model = QuillanRoninOni(cfg).to(device)

# Discover latest/best checkpoint
CKPT_DIR = REPO_ROOT / "checkpoints" / "checkpoints_sft"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
candidates = [
    CKPT_DIR / "quillan_frontier_v2_best.pt",
    CKPT_DIR / "quillan_frontier_v2_latest.pt",
    REPO_ROOT / "checkpoints" / "production_export" / "quillan_ronin_v531_sovereign_production.pt",
    REPO_ROOT / "checkpoints" / "checkpoints_oni" / "quillan_oni_weights.pt",
    REPO_ROOT / "checkpoints" / "checkpoints_oni" / "quillan_oni_latest.pt",
]

loaded_ckpt = None
start_step = 1
best_loss = float("inf")

for c in candidates:
    if c.exists():
        try:
            print(f"[2/2] Loading checkpoint weights from: {c.name}")
            ckpt_data = torch.load(str(c), map_location=device, weights_only=False)
            sd = ckpt_data.get("model_state_dict", ckpt_data.get("model", ckpt_data))
            missing, unexpected = model.load_state_dict(sd, strict=False)
            print(f"[OK] Checkpoint loaded successfully: {len(sd) - len(unexpected)} matching tensors, {len(missing)} missing, {len(unexpected)} unexpected.")
            start_step = int(ckpt_data.get("step", 1))
            best_loss = float(ckpt_data.get("loss", float("inf")))
            loaded_ckpt = c
            break
        except Exception as e:
            print(f"[Warning] Could not load {c.name} ({type(e).__name__}: {e}). Checking next candidate...")

if loaded_ckpt is None:
    print("[Notice] No valid pretrained checkpoint loaded; model is initialized and ready to train from current state.")

print(f"[OK] Model ready on {device}! Starting Step: {start_step}, Best Loss: {best_loss:.4f}")


In [ ]:
# [Cell 3] High-Density Anchor Curriculum (Fast Sub-3.0 Convergence)
DATA_DIR = REPO_ROOT / "training_data"
MAX_SEQ_LEN = 256

def encode_qa_sample(prompt: str, response: str) -> Optional[Tuple[torch.Tensor, torch.Tensor]]:
    p_ids = tokenizer.encode(f"<|user|>\n{prompt.strip()}\n<|assistant|>\n")
    r_ids = tokenizer.encode(f"{response.strip()}<|im_end|>")
    seq = p_ids + r_ids
    labels = [-100] * len(p_ids) + list(r_ids)
    if len(seq) > MAX_SEQ_LEN:
        if MAX_SEQ_LEN - len(p_ids) < 15:
            return None
        seq = seq[:MAX_SEQ_LEN]
        labels = labels[:MAX_SEQ_LEN]
    pad = MAX_SEQ_LEN - len(seq)
    inp = seq + [50256] * pad
    lbl = labels + [-100] * pad
    if sum(1 for l in lbl if l != -100) < 5:
        return None
    return torch.tensor(inp, dtype=torch.long), torch.tensor(lbl, dtype=torch.long)

# Load Curated High-Density Anchor Datasets
anchors = []

# 1. Direct Factual & Syllogism Anchors
direct_path = DATA_DIR / "Quillan_Direct_Answers_Gold.jsonl"
if direct_path.exists():
    with open(direct_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
                pair = encode_qa_sample(d.get("prompt", ""), d.get("response", ""))
                if pair: anchors.append(pair)
            except Exception:
                pass

# 2. Sovereign Thinking Gold (<think> reasoning chains)
think_path = DATA_DIR / "sovereign_thinking_gold.jsonl"
if think_path.exists():
    with open(think_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
                pair = encode_qa_sample(d.get("prompt", ""), d.get("response", ""))
                if pair: anchors.append(pair)
            except Exception:
                pass

# Built-in High-Density Reasoning Fallbacks (Ensures training always works immediately on Colab)
if len(anchors) < 10:
    fallback_anchors = [
        ("If all humans are mortal and Socrates is human, is Socrates mortal? Explain step by step.",
         "<think>\n1. Premise 1: All humans are mortal.\n2. Premise 2: Socrates is a human.\n3. Deduction: Since Socrates belongs to the set of humans, the property of mortality applies to Socrates.\n</think>\nYes, Socrates is mortal. By classical deductive logic (modus ponens), the universal premise that all humans are mortal applies directly to Socrates."),
        ("Explain the Second Law of Thermodynamics with a real-world example.",
         "<think>\nIdentify the core principle of entropy increase in isolated systems.\nSelect a clear physical example such as heat flowing from a hot cup of coffee to the room.\n</think>\nThe Second Law of Thermodynamics states that the total entropy of an isolated system always increases over time. For example, a hot cup of coffee left on a table spontaneously cools down as heat disperses into the cooler room; the heat never spontaneously flows back from the room into the cup."),
        ("Write a Python function to perform binary search on a sorted list.",
         "```python\ndef binary_search(arr, target):\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        mid = (left + right) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n    return -1\n```"),
        ("What is the difference between synchronous and asynchronous execution?",
         "In synchronous execution, tasks are processed sequentially; each task must complete before the next one starts, blocking further operations. In asynchronous execution, tasks can start and run independently without blocking the main execution thread, allowing other operations to continue while waiting for long-running I/O or network responses.")
    ] * 25
    for p, r in fallback_anchors:
        pair = encode_qa_sample(p, r)
        if pair: anchors.append(pair)

print(f"[OK] Loaded {len(anchors)} High-Density Anchor samples for rapid convergence!")


In [ ]:
# [Cell 4] Interactive Training Loop with Live In-Cell Loss Tracking
TRAIN_STEPS = 100          # Adjust number of steps per interactive run
BATCH_SIZE = 2             # Batch size
ACCUM_STEPS = 4            # Gradient accumulation (Smooths gradient variance)
LEARNING_RATE = 0.012      # Muon learning rate

optimizer = create_quillan_muonk2_optimizer(
    model,
    lr_muon=LEARNING_RATE,
    lr_adamw=1e-4,
    weight_decay=0.01,
    ccrl_limit=4.0,
)

model.train()
print(f"Starting Interactive Training: {TRAIN_STEPS} steps (EffBatch={BATCH_SIZE*ACCUM_STEPS})...")
print("-" * 75)

step_losses = []
t0 = time.time()
N_anchors = len(anchors)

for step in range(start_step, start_step + TRAIN_STEPS):
    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    
    for _ in range(ACCUM_STEPS):
        batch_inp, batch_lbl = [], []
        for _ in range(BATCH_SIZE):
            idx = (step * BATCH_SIZE * ACCUM_STEPS) % N_anchors
            inp, lbl = anchors[idx]
            batch_inp.append(inp)
            batch_lbl.append(lbl)
            
        inp_t = torch.stack(batch_inp).to(device)
        lbl_t = torch.stack(batch_lbl).to(device)
        
        out = model(inp_t, labels=lbl_t)
        if isinstance(out, tuple):
            if len(out) == 3:
                logits, ce, aux = out
                loss = ce + (model.total_aux_loss(aux) if hasattr(model, "total_aux_loss") else (aux if isinstance(aux, torch.Tensor) else 0.0))
            elif len(out) == 2:
                logits, loss = out
            else:
                logits, loss = out[0], out[1]
        else:
            logits, loss = out, None
        
        (loss / ACCUM_STEPS).backward()
        accum_loss += loss.item() / ACCUM_STEPS

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    step_losses.append(accum_loss)
    speed = 1.0 / max(time.time() - t0, 1e-4)
    t0 = time.time()
    
    if accum_loss < best_loss:
        best_loss = accum_loss
        save_path = CKPT_DIR / "quillan_frontier_v2_best.pt"
        torch.save({"model_state_dict": model.state_dict(), "step": step, "loss": best_loss}, str(save_path))
        print(f"  * Step [{step:4d}] | Loss: {accum_loss:.4f} | Speed: {speed:.2f} step/s | [NEW BEST] Saved!", flush=True)
    else:
        print(f"  * Step [{step:4d}] | Loss: {accum_loss:.4f} | Speed: {speed:.2f} step/s", flush=True)

# Save latest
latest_path = CKPT_DIR / "quillan_frontier_v2_latest.pt"
torch.save({"model_state_dict": model.state_dict(), "step": start_step + TRAIN_STEPS, "loss": step_losses[-1]}, str(latest_path))
print("-" * 75, flush=True)
print(f"[OK] Training block completed! Final Step Loss: {step_losses[-1]:.4f}, Best Loss: {best_loss:.4f}", flush=True)


In [ ]:
# [Cell 5] Interactive Prompt Testing & Generation Probe
@torch.no_grad()
def ask_quillan(prompt: str, max_tokens: int = 150, temperature: float = 0.7) -> str:
    model.eval()
    formatted = f"<|user|>\n{prompt.strip()}\n<|assistant|>\n"
    tokens = tokenizer.encode(formatted)
    
    gen_tokens = model.generate(
        tokens,
        max_tokens=max_tokens,
        temp=temperature,
        top_k=40,
        top_p=0.90,
    )
    
    gen_text = tokenizer.decode(gen_tokens[len(tokens):])
    gen_text = gen_text.split("<|im_end|>")[0].split("<|endoftext|>")[0].strip()
    return gen_text

# Test on a core logic anchor
test_prompt = "If all humans are mortal and Socrates is human, is Socrates mortal? Explain."
print(f"User: {test_prompt}\n")
response = ask_quillan(test_prompt, max_tokens=100, temperature=0.5)
print(f"Quillan:\n{response}")


### 🎯 Next Steps:
- Re-run **Cell 4** with more steps whenever you want to train another chunk.
- Modify `test_prompt` in **Cell 5** to test any question interactively.
